Maglab calculate predicted phase

In [1]:
import numpy as np
import maglab
import torch.nn.functional as F
import os
import h5py

In [2]:
N = 206
dx = 1.06e-9
Ms = 3.84e5
geo = maglab.geo.cylider(136, 111)
nx, ny, nz = geo.shape
micro = maglab.Micro(nx, ny, nz, dx)
phasemapper = maglab.PhaseMapper(N, dx, rotation_padding=N).cuda()

gauss_kernel_size = 99
sigma = maglab.preprocess.compute_sigma(gauss_kernel_size)
kernel = maglab.preprocess.gaussian_kernel(gauss_kernel_size, sigma)
kernel_groups = kernel.repeat(8, 1, 1, 1).cuda()
print(sigma)

def apply_filter(phase):
    phase1 = F.conv2d(phase.unsqueeze(0), kernel.repeat(1, 1, 1, 1).cuda(), groups=1, padding=gauss_kernel_size // 2)[0,]
    return (phase-phase1).data.cuda()

def load_spin(file_path):
    state = micro.load_state(f"{file_path}/final.pth")
    return state.spin#.detach().cpu().numpy()

15.2


In [3]:
state_pred = micro.load_state("../recon-scripts/dmi_loss/14_layers_loss/results/loss1.4e+01_wphi1.0e+07/final.pth")
spin_pred = state_pred.spin

/home/zhaoy/anaconda3/envs/tomopy/lib/python3.12/site-packages/maglab-0.1-py3.12.egg/maglab/micro.py:85: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [4]:
output_pred_dir = 'hdf5/pred' 
os.makedirs(output_pred_dir, exist_ok=True)
angles = [0, -5.2, -8.2, -12.2, -16, -20, -24, -28, -32, -36, -40.8, -44, -46.9, -50, -54, -58, -61, -65.2]
for i, angle in enumerate(angles):
    alpha = angle
    phi_pred = apply_filter(phasemapper(spin_pred, alpha=alpha, Ms=Ms))# * mask.cuda()
    output_path = 'phase_{:g}.hdf5'.format(np.abs(angle))
    path = os.path.join(output_pred_dir, output_path)
    with h5py.File(path, 'w') as hf:
        hf.create_dataset('phase', data=phi_pred.detach().cpu().numpy().astype(np.float32))
    print(f"Saved: {path}")

Saved: hdf5/pred/phase_0.hdf5
Saved: hdf5/pred/phase_5.2.hdf5
Saved: hdf5/pred/phase_8.2.hdf5
Saved: hdf5/pred/phase_12.2.hdf5
Saved: hdf5/pred/phase_16.hdf5
Saved: hdf5/pred/phase_20.hdf5
Saved: hdf5/pred/phase_24.hdf5
Saved: hdf5/pred/phase_28.hdf5
Saved: hdf5/pred/phase_32.hdf5
Saved: hdf5/pred/phase_36.hdf5
Saved: hdf5/pred/phase_40.8.hdf5
Saved: hdf5/pred/phase_44.hdf5
Saved: hdf5/pred/phase_46.9.hdf5
Saved: hdf5/pred/phase_50.hdf5
Saved: hdf5/pred/phase_54.hdf5
Saved: hdf5/pred/phase_58.hdf5
Saved: hdf5/pred/phase_61.hdf5
Saved: hdf5/pred/phase_65.2.hdf5


Pyramid plotting phase image

In [ ]:
import sys
sys.path.extend(['/home/zhaoy/packages/Pyramid']) 
#sys.path.extend(['/Users/fzheng/SkyDrive/Codes/Python/Pyramid']) # where Pyramid situates
print('Python %s on %s' % (sys.version, sys.platform))

import numpy as np
import matplotlib.pyplot as plt 
%matplotlib inline
import pyramid as pr
from scipy.ndimage import rotate, gaussian_filter
pr.plottools.pretty_plots()
import h5py

angles = [0, 5.2, 8.2, 12.2, 16, 20, 24, 28, 32, 36, 40.8, 44, 46.9, 50, 54, 58, 61, 65.2]
fig = plt.figure(figsize=(18, 36))
ax = {}
for i, angle in enumerate(angles):
    phase_name = './hdf5/pred/phase_{:g}.hdf5'.format(angle)
    with h5py.File(phase_name, 'r') as hf:
        phase = np.fliplr(rotate(hf['phase'][:], angle=90, reshape=False, order=1))
        phase = pr.PhaseMap(a=1, phase=phase)
        phase.phase = gaussian_filter(phase.phase, sigma=0) 
    tilt = angle
    ax[i+1] = fig.add_subplot(6, 3, i+1, aspect='equal')
    ax[i+1].get_xaxis().set_visible(False)
    ax[i+1].get_yaxis().set_visible(False)
    phase.plot_holo(axis=ax[i+1],gain=40,scalebar=False,colorwheel=True,note="{}".format(''))
    #phase.plot_phase(axis=ax[i+1],scalebar=False,note="{}".format(''),cbar=False,show_mask=True,cmap='gray',vmax=0.5,vmin=-0.5)
    plt.tight_layout(h_pad=0.4)

Python 3.5.6 |Anaconda, Inc.| (default, Jun  4 2021, 13:57:47) 
[GCC 7.5.0] on linux


/home/zhaoy/anaconda3/envs/pyramid/lib/python3.5/site-packages/matplotlib/figure.py:1742: UserWarning: This figure includes Axes that are not compatible with tight_layout, so its results might be incorrect.
  warnings.warn("This figure includes Axes that are not "
